# BO-Guided Center Adaptation with Paper-Inspired DORAEMON (TinySim Mujoco Manipulation)

This notebook adapts the DORAEMON-paper-inspired BO pipeline to a **Mujoco manipulation** task:
- **Surrogate**: TinySim Mujoco `ManipulationEnvV0` with domain-randomised **friction** and **mass**
- **Target**: Same environment with hidden fixed target physics parameters
- Bounded **Beta** distribution over DR parameters
- **Entropy maximisation** under a success constraint
- **KL trust-region** between consecutive distribution updates
- **SAC** agent (continuous actions) replaces DQN

## 1) Imports and Global Setup

In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import time
import copy
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

import mujoco
from tinysim_mujoco.manipulation import ManipulationBaseEnv
from tinysim_mujoco.manipulation.push_env import ManipulationEnvV0
from scipy.special import betaln, digamma

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")
print("mujoco:", mujoco.__version__)

In [ ]:
try:
    from skopt import Optimizer
    from skopt.space import Real
except Exception as exc:
    raise ImportError(
        "scikit-optimize is required.  pip install scikit-optimize"
    ) from exc

print("skopt imported OK")

## 2) Config

- `TARGET_PARAMS` is used **only** for target evaluation.
- BO never gets direct access to target params beyond scalar target eval score.
- DR parameters: **friction** (object surface) and **mass** (object mass).

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

PROJECT_DIR = Path(".").resolve()
RUNS_ROOT = PROJECT_DIR / "runs_manipulation"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

WARMSTART_CHECKPOINT = PROJECT_DIR / "manipulation_saved" / "manipulation_sac.pt"

# Domain-randomisation bounds for (friction, mass)
DR_BOUNDS = {
    "friction": (0.1, 2.5),
    "mass":     (0.02, 0.50),
}
EDGE_DR_BOUNDS = {
    "friction": (0.2, 1.5),
    "mass":     (0.05, 0.40),
}

# Initial centre (default env values)
MU0 = np.array([1.0, 0.1], dtype=np.float32)
MU0_EDGE = np.array([
    float(np.clip(MU0[0], EDGE_DR_BOUNDS["friction"][0], EDGE_DR_BOUNDS["friction"][1])),
    float(np.clip(MU0[1], EDGE_DR_BOUNDS["mass"][0], EDGE_DR_BOUNDS["mass"][1])),
], dtype=np.float32)

# Hidden target parameters (only used for evaluation)
TARGET_PARAMS = {
    "friction": 0.4,
    "mass":     0.30,
}

MAX_EPISODE_STEPS = 128

# DORAEMON hyper-parameters
ALPHA_TARGET         = 0.60
KL_MAX               = 0.04
C_INIT               = np.array([18.0, 18.0], dtype=np.float32)
C_MIN                = np.array([2.2, 2.2],   dtype=np.float32)
C_MAX                = np.array([220.0, 220.0], dtype=np.float32)
CANDIDATES_PER_UPDATE = 300
MEAN_PERTURB_STD     = 0.06
LOGC_PERTURB_STD     = 0.20
MIN_WEIGHT_LOG       = -20.0
MAX_WEIGHT_LOG       =  20.0

FREEZE_DORAEMON_MEAN = True
CONC_INIT_MODE       = "reset"    # reset | carry | blend
CONC_BLEND           = 0.5

# Budget presets
SMOKE_CFG = dict(T=3,  blocks=5,  episodes_per_block=10, B_r=5)
PROTO_CFG = dict(T=8,  blocks=10, episodes_per_block=15, B_r=10)
EDGE_CFG  = dict(T=15, blocks=15, episodes_per_block=20, B_r=15)

PRETRAIN_EPISODES = 500

print("Target params (eval only):", TARGET_PARAMS)
print("MU0_EDGE:", MU0_EDGE.tolist())
print("FREEZE_DORAEMON_MEAN:", FREEZE_DORAEMON_MEAN)

## 3) SAC Components (Manipulation)

In [ ]:
LOG_STD_MIN = -20.0
LOG_STD_MAX = 2.0


class ContinuousReplayBuffer:
    def __init__(self, capacity: int = 100_000):
        self.capacity = int(capacity)
        self.buffer: list = []
        self.pos = 0

    def push(self, state, action, reward, next_state, done):
        item = (
            np.asarray(state, dtype=np.float32),
            np.asarray(action, dtype=np.float32),
            float(reward),
            np.asarray(next_state, dtype=np.float32),
            float(done),
        )
        if len(self.buffer) < self.capacity:
            self.buffer.append(item)
        else:
            self.buffer[self.pos] = item
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size: int, device: str):
        idx = np.random.choice(len(self.buffer), size=batch_size, replace=False)
        batch = [self.buffer[i] for i in idx]
        s, a, r, ns, d = zip(*batch)
        return (
            torch.tensor(np.stack(s),  dtype=torch.float32, device=device),
            torch.tensor(np.stack(a),  dtype=torch.float32, device=device),
            torch.tensor(r,  dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.stack(ns), dtype=torch.float32, device=device),
            torch.tensor(d,  dtype=torch.float32, device=device).unsqueeze(1),
        )

    def __len__(self):
        return len(self.buffer)


class GaussianActor(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mean_head    = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)

    def forward(self, state):
        x = self.net(state)
        mean    = self.mean_head(x)
        log_std = self.log_std_head(x).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def sample(self, state):
        mean, log_std = self.forward(state)
        std  = log_std.exp()
        dist = Normal(mean, std)
        x_t  = dist.rsample()
        action   = torch.tanh(x_t)
        log_prob = dist.log_prob(x_t) - torch.log(1.0 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        return action, log_prob

    def deterministic(self, state):
        mean, _ = self.forward(state)
        return torch.tanh(mean)


class TwinQNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 256):
        super().__init__()
        inp = state_dim + action_dim
        self.q1 = nn.Sequential(
            nn.Linear(inp, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(inp, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state, action):
        sa = torch.cat([state, action], dim=-1)
        return self.q1(sa), self.q2(sa)


@dataclass
class SACConfig:
    state_dim:       int   = 15    # 12 obs + 3 goal
    action_dim:      int   = 3     # EE delta XYZ
    hidden_dim:      int   = 256
    actor_lr:        float = 3e-4
    critic_lr:       float = 3e-4
    alpha_lr:        float = 3e-4
    gamma:           float = 0.98
    tau:             float = 0.005
    buffer_capacity: int   = 100_000
    batch_size:      int   = 256
    init_alpha:      float = 0.2
    auto_alpha:      bool  = True


class SACAgent:
    def __init__(self, cfg: SACConfig, device: str = "cpu"):
        self.cfg    = cfg
        self.device = device

        self.actor = GaussianActor(cfg.state_dim, cfg.action_dim, cfg.hidden_dim).to(device)
        self.critic = TwinQNetwork(cfg.state_dim, cfg.action_dim, cfg.hidden_dim).to(device)
        self.critic_target = TwinQNetwork(cfg.state_dim, cfg.action_dim, cfg.hidden_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.actor_optimizer  = optim.Adam(self.actor.parameters(),  lr=cfg.actor_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=cfg.critic_lr)

        self.log_alpha = torch.tensor(
            np.log(cfg.init_alpha), dtype=torch.float32, device=device, requires_grad=True
        )
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=cfg.alpha_lr)
        self.target_entropy  = -float(cfg.action_dim)

        self.memory = ContinuousReplayBuffer(cfg.buffer_capacity)
        self.training_losses: list[float] = []
        self._episode_counter = 0

    @property
    def alpha(self) -> float:
        return self.log_alpha.exp().item()

    # ── act ──
    def act(self, state: np.ndarray, training: bool = True) -> np.ndarray:
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            if training:
                action, _ = self.actor.sample(s)
            else:
                action = self.actor.deterministic(s)
            return action.squeeze(0).cpu().numpy()

    def store(self, s, a, r, ns, done):
        self.memory.push(s, a, r, ns, done)

    # ── learn ──
    def learn_step(self) -> float | None:
        if len(self.memory) < self.cfg.batch_size:
            return None
        s, a, r, ns, d = self.memory.sample(self.cfg.batch_size, self.device)
        alpha = self.log_alpha.exp().detach()

        # critic
        with torch.no_grad():
            na, nlp = self.actor.sample(ns)
            q1n, q2n = self.critic_target(ns, na)
            q_target = r + self.cfg.gamma * (1.0 - d) * (torch.min(q1n, q2n) - alpha * nlp)
        q1, q2 = self.critic(s, a)
        critic_loss = nn.MSELoss()(q1, q_target) + nn.MSELoss()(q2, q_target)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # actor
        new_a, log_prob = self.actor.sample(s)
        q1_new, q2_new = self.critic(s, new_a)
        actor_loss = (alpha * log_prob - torch.min(q1_new, q2_new)).mean()
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # alpha
        if self.cfg.auto_alpha:
            alpha_loss = -(self.log_alpha * (log_prob.detach() + self.target_entropy)).mean()
            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()

        # soft-update target critic
        for p, tp in zip(self.critic.parameters(), self.critic_target.parameters()):
            tp.data.copy_(self.cfg.tau * p.data + (1.0 - self.cfg.tau) * tp.data)

        loss_val = float(critic_loss.item())
        self.training_losses.append(loss_val)
        return loss_val

    def end_episode(self):
        self._episode_counter += 1

    # ── serialisation ──
    def state_payload(self) -> dict[str, Any]:
        return {
            "actor_state_dict":         self.actor.state_dict(),
            "critic_state_dict":        self.critic.state_dict(),
            "critic_target_state_dict": self.critic_target.state_dict(),
            "actor_optimizer":          self.actor_optimizer.state_dict(),
            "critic_optimizer":         self.critic_optimizer.state_dict(),
            "log_alpha":                self.log_alpha.detach().cpu().item(),
            "alpha_optimizer":          self.alpha_optimizer.state_dict(),
            "training_losses":          self.training_losses[-1000:],
            "episode_counter":          self._episode_counter,
            "hparams":                  vars(self.cfg),
        }

    def load_payload(self, payload: dict[str, Any]):
        self.actor.load_state_dict(payload["actor_state_dict"])
        self.critic.load_state_dict(payload["critic_state_dict"])
        self.critic_target.load_state_dict(
            payload.get("critic_target_state_dict", payload["critic_state_dict"])
        )
        if "actor_optimizer" in payload:
            self.actor_optimizer.load_state_dict(payload["actor_optimizer"])
        if "critic_optimizer" in payload:
            self.critic_optimizer.load_state_dict(payload["critic_optimizer"])
        if "log_alpha" in payload:
            self.log_alpha.data.fill_(float(payload["log_alpha"]))
        if "alpha_optimizer" in payload:
            self.alpha_optimizer.load_state_dict(payload["alpha_optimizer"])
        self.training_losses = list(payload.get("training_losses", []))
        self._episode_counter = int(payload.get("episode_counter", 0))

## 4) Warm-start and Checkpoint Helpers

In [ ]:
def make_agent_from_hparams(hparams: dict | None, device: str = DEVICE) -> SACAgent:
    if hparams is None:
        cfg = SACConfig()
    else:
        cfg = SACConfig(**{k: v for k, v in hparams.items() if k in SACConfig.__annotations__})
    return SACAgent(cfg=cfg, device=device)


def load_warmstart_agent(checkpoint_path: str | Path, device: str = DEVICE) -> SACAgent:
    payload = torch.load(str(checkpoint_path), map_location=device, weights_only=False)
    hparams = payload.get("hparams")
    agent = make_agent_from_hparams(hparams, device=device)
    agent.load_payload(payload)
    return agent


def save_agent_checkpoint(agent: SACAgent, path: str | Path, extra: dict | None = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = agent.state_payload()
    payload["extra"] = extra or {}
    torch.save(payload, str(path))


def clone_agent(agent: SACAgent) -> SACAgent:
    cloned = make_agent_from_hparams(vars(agent.cfg), device=agent.device)
    cloned.load_payload(agent.state_payload())
    return cloned

## 5) Environment and Evaluation Helpers

In [ ]:
# ── environment factory ──

def make_manipulation_env(friction: float, mass: float) -> ManipulationEnvV0:
    env = ManipulationEnvV0(headless=True)
    # modify object friction
    geom_id = mujoco.mj_name2id(env._model, mujoco.mjtObj.mjOBJ_GEOM, "obj_geom")
    env._model.geom_friction[geom_id, 0] = float(friction)
    # modify object mass + recompute box inertia (half-size 0.03)
    body_id = mujoco.mj_name2id(env._model, mujoco.mjtObj.mjOBJ_BODY, "obj")
    env._model.body_mass[body_id] = float(mass)
    s = 0.03  # box half-size from table.xml
    inertia_diag = mass / 3.0 * (s**2 + s**2)
    env._model.body_inertia[body_id] = np.array([inertia_diag, inertia_diag, inertia_diag])
    mujoco.mj_forward(env._model, env._data)
    return env


def flatten_obs(obs_dict: dict) -> np.ndarray:
    return np.concatenate([obs_dict["observation"], obs_dict["desired_goal"]]).astype(np.float32)


# ── Beta distribution helpers ──

def normalize_params(friction: float, mass: float, bounds: dict) -> np.ndarray:
    zf = (float(friction) - bounds["friction"][0]) / (bounds["friction"][1] - bounds["friction"][0])
    zm = (float(mass) - bounds["mass"][0]) / (bounds["mass"][1] - bounds["mass"][0])
    return np.array([zf, zm], dtype=np.float64)


def denormalize_params(z: np.ndarray, bounds: dict) -> tuple[float, float]:
    friction = bounds["friction"][0] + float(z[0]) * (bounds["friction"][1] - bounds["friction"][0])
    mass     = bounds["mass"][0]     + float(z[1]) * (bounds["mass"][1]     - bounds["mass"][0])
    return float(friction), float(mass)


def beta_from_mean_conc(mean: np.ndarray, conc: np.ndarray):
    m = np.clip(np.asarray(mean, dtype=np.float64), 1e-4, 1.0 - 1e-4)
    c = np.clip(np.asarray(conc, dtype=np.float64), 2.0001, None)
    return m * c, (1.0 - m) * c


def mean_conc_from_beta(alpha: np.ndarray, beta: np.ndarray):
    a = np.asarray(alpha, dtype=np.float64)
    b = np.asarray(beta, dtype=np.float64)
    c = a + b
    return a / np.clip(c, 1e-8, None), c


def make_beta_dist_at_mu(mu: np.ndarray, conc: np.ndarray, bounds: dict) -> dict:
    mu_c = np.array([
        float(np.clip(mu[0], bounds["friction"][0], bounds["friction"][1])),
        float(np.clip(mu[1], bounds["mass"][0], bounds["mass"][1])),
    ], dtype=np.float64)
    z = normalize_params(mu_c[0], mu_c[1], bounds)
    a, b = beta_from_mean_conc(z, conc)
    return {"alpha": a, "beta": b, "bounds": bounds}


def beta_logpdf(z, alpha, beta):
    z = np.clip(np.asarray(z, dtype=np.float64), 1e-6, 1.0 - 1e-6)
    a = np.asarray(alpha, dtype=np.float64)
    b = np.asarray(beta, dtype=np.float64)
    return (a - 1.0) * np.log(z) + (b - 1.0) * np.log(1.0 - z) - betaln(a, b)


def beta_entropy_fn(alpha, beta):
    a = np.asarray(alpha, dtype=np.float64)
    b = np.asarray(beta, dtype=np.float64)
    return betaln(a, b) - (a - 1) * digamma(a) - (b - 1) * digamma(b) + (a + b - 2) * digamma(a + b)


def dist_entropy(dist: dict) -> float:
    h = beta_entropy_fn(dist["alpha"], dist["beta"])
    s1 = np.log(dist["bounds"]["friction"][1] - dist["bounds"]["friction"][0])
    s2 = np.log(dist["bounds"]["mass"][1] - dist["bounds"]["mass"][0])
    return float(np.sum(h) + s1 + s2)


def beta_kl(a0, b0, a1, b1):
    return (betaln(a1, b1) - betaln(a0, b0)
            + (a0 - a1) * digamma(a0) + (b0 - b1) * digamma(b0)
            + (a1 + b1 - a0 - b0) * digamma(a0 + b0))


def dist_kl(old: dict, new: dict) -> float:
    return float(np.sum(beta_kl(old["alpha"], old["beta"], new["alpha"], new["beta"])))


def sample_beta_params(dist: dict, rng: np.random.Generator):
    z = rng.beta(dist["alpha"], dist["beta"])
    z = np.clip(z, 1e-6, 1.0 - 1e-6)
    friction, mass = denormalize_params(z, dist["bounds"])
    return friction, mass, z.astype(np.float64)


# ── importance-sampling success estimate ──

def is_success_estimate(z_samples, success, old_dist, new_dist, min_log_w=-20.0, max_log_w=20.0):
    if len(z_samples) == 0:
        return 0.0
    z = np.clip(np.asarray(z_samples, dtype=np.float64), 1e-6, 1.0 - 1e-6)
    s = np.asarray(success, dtype=np.float64)
    logp_old = np.sum(beta_logpdf(z, old_dist["alpha"], old_dist["beta"]), axis=1)
    logp_new = np.sum(beta_logpdf(z, new_dist["alpha"], new_dist["beta"]), axis=1)
    logw = np.clip(logp_new - logp_old, min_log_w, max_log_w)
    w = np.exp(logw - np.max(logw))
    w = w / np.clip(np.sum(w), 1e-12, None)
    return float(np.sum(w * s))


# ── DORAEMON update ──

def doraemon_update_beta(
    old_dist, z_samples, success_samples,
    alpha_target, kl_max, rng,
    n_candidates, mean_std, logc_std,
    c_min, c_max, min_log_w, max_log_w,
    freeze_mean=True,
):
    m_old, c_old = mean_conc_from_beta(old_dist["alpha"], old_dist["beta"])
    candidates = [(m_old.copy(), c_old.copy())]
    for _ in range(int(n_candidates)):
        m = m_old.copy() if freeze_mean else np.clip(m_old + rng.normal(0, mean_std, size=2), 1e-4, 1 - 1e-4)
        c = np.exp(np.log(np.clip(c_old, 1e-8, None)) + rng.normal(0, logc_std, size=2))
        c = np.clip(c, c_min, c_max)
        candidates.append((m, c))

    best_feasible = None
    backup = None
    for m, c in candidates:
        a_new, b_new = beta_from_mean_conc(m, c)
        cand = {"alpha": a_new, "beta": b_new, "bounds": old_dist["bounds"]}
        kl = dist_kl(old_dist, cand)
        if kl > kl_max:
            continue
        g_hat = is_success_estimate(z_samples, success_samples, old_dist, cand, min_log_w, max_log_w)
        ent = dist_entropy(cand)
        if backup is None or g_hat > backup["g_hat"] or (np.isclose(g_hat, backup["g_hat"]) and ent > backup["entropy"]):
            backup = {"dist": cand, "g_hat": float(g_hat), "entropy": float(ent), "kl": float(kl), "mode": "backup_success"}
        if g_hat >= alpha_target:
            if best_feasible is None or ent > best_feasible["entropy"]:
                best_feasible = {"dist": cand, "g_hat": float(g_hat), "entropy": float(ent), "kl": float(kl), "mode": "feasible_entropy"}

    if best_feasible is not None:
        return best_feasible["dist"], best_feasible
    if backup is not None:
        return backup["dist"], backup
    return old_dist, {"dist": old_dist, "g_hat": 0.0, "entropy": dist_entropy(old_dist), "kl": 0.0, "mode": "no_candidate"}


# ── target evaluation ──

def target_eval_manipulation(agent, n_eval, max_steps, target_params, seed):
    successes = 0
    for ep in range(n_eval):
        env = make_manipulation_env(friction=target_params["friction"], mass=target_params["mass"])
        obs_dict, _ = env.reset()
        obs = flatten_obs(obs_dict)
        solved = False
        for _ in range(max_steps):
            a = agent.act(obs, training=False)
            obs_dict, reward, terminated, truncated, _ = env.step(a)
            obs = flatten_obs(obs_dict)
            if terminated:
                solved = True
                break
            if truncated:
                break
        successes += int(solved)
    rate = float(successes / max(1, n_eval))
    return rate, int(successes), int(n_eval)

## 6) DORAEMON Training Under DR

In [ ]:
def train_under_doraemon_beta(
    agent, mu, conc_init,
    blocks, episodes_per_block,
    bounds, max_steps,
    alpha_target, kl_max,
    n_candidates, mean_std, logc_std,
    c_min, c_max,
    min_log_w, max_log_w,
    seed, freeze_mean=True,
):
    rng = np.random.default_rng(seed)
    dist = make_beta_dist_at_mu(mu, conc_init, bounds)

    g_history = []
    conc_history = []
    block_history = []

    for b in range(int(blocks)):
        z_batch, s_batch, loss_batch = [], [], []

        for e in range(int(episodes_per_block)):
            friction_e, mass_e, z_e = sample_beta_params(dist, rng)
            env = make_manipulation_env(friction=friction_e, mass=mass_e)
            obs_dict, _ = env.reset()
            obs = flatten_obs(obs_dict)
            losses = []
            solved = False

            for _ in range(max_steps):
                a = agent.act(obs, training=True)
                obs_dict, reward, terminated, truncated, _ = env.step(a)
                next_obs = flatten_obs(obs_dict)
                done = terminated or truncated
                agent.store(obs, a, reward, next_obs, float(done))
                loss = agent.learn_step()
                if loss is not None:
                    losses.append(loss)
                obs = next_obs
                if terminated:
                    solved = True
                    break
                if truncated:
                    break

            agent.end_episode()
            z_batch.append(z_e)
            s_batch.append(1.0 if solved else 0.0)
            if losses:
                loss_batch.append(float(np.mean(losses)))

        z_batch = np.asarray(z_batch, dtype=np.float64)
        s_batch = np.asarray(s_batch, dtype=np.float64)

        dist, upd = doraemon_update_beta(
            old_dist=dist, z_samples=z_batch, success_samples=s_batch,
            alpha_target=alpha_target, kl_max=kl_max, rng=rng,
            n_candidates=n_candidates, mean_std=mean_std, logc_std=logc_std,
            c_min=np.asarray(c_min, dtype=np.float64),
            c_max=np.asarray(c_max, dtype=np.float64),
            min_log_w=min_log_w, max_log_w=max_log_w,
            freeze_mean=freeze_mean,
        )

        _, conc_end = mean_conc_from_beta(dist["alpha"], dist["beta"])
        g_history.append(float(upd["g_hat"]))
        conc_history.append(conc_end.copy())

        m_end, _ = mean_conc_from_beta(dist["alpha"], dist["beta"])
        mu_end = denormalize_params(m_end, bounds)
        block_history.append({
            "block": int(b + 1),
            "mode": str(upd["mode"]),
            "emp_success": float(np.mean(s_batch)) if len(s_batch) else 0.0,
            "g_hat": float(upd["g_hat"]),
            "entropy": float(upd["entropy"]),
            "kl": float(upd["kl"]),
            "mu_friction_end": float(mu_end[0]),
            "mu_mass_end": float(mu_end[1]),
            "conc_friction_end": float(conc_end[0]),
            "conc_mass_end": float(conc_end[1]),
            "loss_mean": float(np.mean(loss_batch)) if loss_batch else float("nan"),
        })

    return agent, g_history, conc_history, dist, block_history

## 7) BO + DORAEMON Main Loop

In [ ]:
def project_mu_to_bounds(mu, bounds):
    return np.array([
        float(np.clip(mu[0], bounds["friction"][0], bounds["friction"][1])),
        float(np.clip(mu[1], bounds["mass"][0], bounds["mass"][1])),
    ], dtype=np.float32)


def _save_run_config(run_dir, **kwargs):
    """Dump every hyperparameter to config.json at the start of a run."""
    config = {}
    for k, v in kwargs.items():
        if isinstance(v, np.ndarray):
            config[k] = v.tolist()
        elif isinstance(v, Path):
            config[k] = str(v)
        else:
            config[k] = v
    config["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S")
    config["device"] = DEVICE
    config["seed_global"] = SEED
    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)


def run_bo_doraemon_paperlike(
    checkpoint_path, mu0, conc0,
    T, blocks, episodes_per_block, B_r,
    bounds, max_steps,
    alpha_target, kl_max,
    n_candidates, mean_std, logc_std,
    c_min, c_max, min_log_w, max_log_w,
    target_params, seed=42,
    freeze_doraemon_mean=True,
    conc_init_mode="reset", conc_blend=0.5,
):
    run_name = time.strftime("bo_dora_manip_%Y%m%d_%H%M%S")
    run_dir  = RUNS_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # ── save full config for reproducibility ──
    _save_run_config(
        run_dir,
        checkpoint_path=checkpoint_path,
        mu0=mu0, conc0=conc0,
        T=T, blocks=blocks, episodes_per_block=episodes_per_block, B_r=B_r,
        bounds=bounds, max_steps=max_steps,
        alpha_target=alpha_target, kl_max=kl_max,
        n_candidates=n_candidates, mean_std=mean_std, logc_std=logc_std,
        c_min=c_min, c_max=c_max,
        min_log_w=min_log_w, max_log_w=max_log_w,
        target_params=target_params, seed=seed,
        freeze_doraemon_mean=freeze_doraemon_mean,
        conc_init_mode=conc_init_mode, conc_blend=conc_blend,
    )

    hist_csv   = run_dir / "history.csv"
    hist_json  = run_dir / "history.json"
    block_json = run_dir / "block_history.json"
    best_ckpt  = run_dir / "best_policy.pt"
    last_ckpt  = run_dir / "last_policy.pt"

    # load or create agent
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        agent_prev = load_warmstart_agent(checkpoint_path, device=DEVICE)
        print(f"[BO init] warm-start loaded from {checkpoint_path}")
    else:
        agent_prev = make_agent_from_hparams(None, device=DEVICE)
        print("[BO init] no checkpoint, starting from scratch")

    conc_prev = np.asarray(conc0, dtype=np.float32).copy()

    opt = Optimizer(
        dimensions=[
            Real(bounds["friction"][0], bounds["friction"][1], name="friction_center"),
            Real(bounds["mass"][0],     bounds["mass"][1],     name="mass_center"),
        ],
        base_estimator="GP", acq_func="EI", random_state=seed,
    )

    history    = []
    block_rows = []

    mu0      = np.array(mu0, dtype=np.float32)
    mu0_proj = project_mu_to_bounds(mu0, bounds)
    if not np.allclose(mu0, mu0_proj):
        print(f"[BO init] mu0 projected: ({mu0[0]:.4f}, {mu0[1]:.4f}) -> ({mu0_proj[0]:.4f}, {mu0_proj[1]:.4f})")

    # initial target eval
    f0, succ0, eps0 = target_eval_manipulation(agent_prev, B_r, max_steps, target_params, seed + 7)
    opt.tell(mu0_proj.tolist(), -f0)

    best_score   = float(f0)
    best_center  = mu0_proj.copy()
    best_payload = copy.deepcopy(agent_prev.state_payload())
    save_agent_checkpoint(agent_prev, best_ckpt, extra={"center": mu0_proj.tolist(), "score": float(f0), "iter": 0})

    history.append({
        "iter": 0,
        "mu_friction": float(mu0_proj[0]), "mu_mass": float(mu0_proj[1]),
        "conc_friction_start": float(conc_prev[0]), "conc_mass_start": float(conc_prev[1]),
        "conc_friction_end":   float(conc_prev[0]), "conc_mass_end":   float(conc_prev[1]),
        "g_mean": float("nan"), "g_last": float("nan"),
        "entropy_last": float("nan"), "kl_last": float("nan"),
        "target_solve_rate": float(f0),
        "target_successes": int(succ0), "target_episodes": int(eps0),
        "is_best": True,
    })
    print(f"[BO init] target={f0:.3f} ({succ0}/{eps0})")

    for t in range(1, T + 1):
        mu_t = np.array(opt.ask(), dtype=np.float32)
        agent_t = clone_agent(agent_prev)

        # concentration init mode
        if conc_init_mode == "carry":
            conc_start = conc_prev.copy()
        elif conc_init_mode == "blend":
            w = float(np.clip(conc_blend, 0.0, 1.0))
            conc_start = (1.0 - w) * np.asarray(conc0, dtype=np.float32) + w * conc_prev
        else:
            conc_start = np.asarray(conc0, dtype=np.float32).copy()

        agent_t, g_hist, conc_hist, dist_end, block_hist = train_under_doraemon_beta(
            agent=agent_t, mu=mu_t, conc_init=conc_start,
            blocks=blocks, episodes_per_block=episodes_per_block,
            bounds=bounds, max_steps=max_steps,
            alpha_target=alpha_target, kl_max=kl_max,
            n_candidates=n_candidates, mean_std=mean_std, logc_std=logc_std,
            c_min=c_min, c_max=c_max,
            min_log_w=min_log_w, max_log_w=max_log_w,
            seed=seed + 10000 * t, freeze_mean=freeze_doraemon_mean,
        )

        for r in block_hist:
            block_rows.append({"iter": int(t), **r})

        f_t, succ_t, eps_t = target_eval_manipulation(agent_t, B_r, max_steps, target_params, seed + 333 + t)
        opt.tell(mu_t.tolist(), -float(f_t))

        conc_end = conc_hist[-1] if conc_hist else conc_start

        is_best = False
        if float(f_t) > best_score:
            best_score   = float(f_t)
            best_center  = mu_t.copy()
            best_payload = copy.deepcopy(agent_t.state_payload())
            save_agent_checkpoint(agent_t, best_ckpt, extra={"center": mu_t.tolist(), "score": float(f_t), "iter": t})
            is_best = True

        row = {
            "iter": t,
            "mu_friction": float(mu_t[0]), "mu_mass": float(mu_t[1]),
            "conc_friction_start": float(conc_start[0]), "conc_mass_start": float(conc_start[1]),
            "conc_friction_end":   float(conc_end[0]),   "conc_mass_end":   float(conc_end[1]),
            "g_mean": float(np.mean(g_hist)) if g_hist else float("nan"),
            "g_last": float(g_hist[-1]) if g_hist else float("nan"),
            "entropy_last": float(block_hist[-1]["entropy"]) if block_hist else float("nan"),
            "kl_last":      float(block_hist[-1]["kl"])      if block_hist else float("nan"),
            "target_solve_rate": float(f_t),
            "target_successes": int(succ_t), "target_episodes": int(eps_t),
            "is_best": bool(is_best),
        }
        history.append(row)

        with open(hist_json, "w") as f:
            json.dump(history, f, indent=2)
        with open(hist_csv, "w", newline="") as f:
            fieldnames = list(dict.fromkeys(k for r in history for k in r.keys()))
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(history)
        with open(block_json, "w") as f:
            json.dump(block_rows, f, indent=2)

        agent_prev = agent_t
        conc_prev  = np.asarray(conc_end, dtype=np.float32).copy()

        print(
            f"[BO step {t:02d}/{T}] mu=({mu_t[0]:.4f}, {mu_t[1]:.4f}) "
            f"g_last={row['g_last']:.3f} target={f_t:.3f} ({succ_t}/{eps_t}) "
            f"conc=({conc_end[0]:.2f}, {conc_end[1]:.2f})"
        )

    save_agent_checkpoint(agent_prev, last_ckpt, extra={"iter": T, "best_score": best_score})

    return {
        "run_dir": str(run_dir),
        "history": history,
        "block_history": block_rows,
        "best_center": best_center.tolist(),
        "best_score": float(best_score),
        "best_payload": best_payload,
        "best_checkpoint": str(best_ckpt),
        "last_checkpoint": str(last_ckpt),
        "history_csv": str(hist_csv),
        "history_json": str(hist_json),
        "block_history_json": str(block_json),
    }

## 8) Pre-train Warm-start Policy

Train an SAC agent on the **default** manipulation env (friction=1.0, mass=0.1) to produce a warm-start checkpoint.  Skip this cell if you already have one.

In [ ]:
def pretrain_manipulation_agent(
    n_episodes=PRETRAIN_EPISODES,
    friction=1.0, mass=0.1,
    max_steps=MAX_EPISODE_STEPS,
    seed=SEED, save_path=None,
):
    agent = make_agent_from_hparams(None, device=DEVICE)
    rewards_log = []

    for ep in range(n_episodes):
        env = make_manipulation_env(friction=friction, mass=mass)
        obs_dict, _ = env.reset()
        obs = flatten_obs(obs_dict)
        ep_reward = 0.0

        for _ in range(max_steps):
            a = agent.act(obs, training=True)
            obs_dict, reward, terminated, truncated, _ = env.step(a)
            next_obs = flatten_obs(obs_dict)
            done = terminated or truncated
            agent.store(obs, a, reward, next_obs, float(done))
            agent.learn_step()
            obs = next_obs
            ep_reward += reward
            if done:
                break

        agent.end_episode()
        rewards_log.append(ep_reward)

        if (ep + 1) % 50 == 0:
            avg = np.mean(rewards_log[-50:])
            print(f"  pretrain ep {ep+1:4d}/{n_episodes} | avg_reward(50)={avg:.2f} | alpha={agent.alpha:.4f}")

    if save_path is not None:
        save_agent_checkpoint(agent, save_path, extra={"pretrain_episodes": n_episodes})
        print(f"Saved warm-start checkpoint to {save_path}")
    return agent


if not WARMSTART_CHECKPOINT.exists():
    WARMSTART_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    print(f"No warm-start found at {WARMSTART_CHECKPOINT}, pre-training...")
    _agent = pretrain_manipulation_agent(save_path=WARMSTART_CHECKPOINT)
    del _agent
else:
    print(f"Warm-start checkpoint exists: {WARMSTART_CHECKPOINT}")

## 8b) Sanity Check: Visualise Rollouts at Different Physics

Run a few random-action episodes at the **default**, **target**, and **extreme** physics settings.
This lets you verify that friction/mass changes produce visible behavioural differences
before committing to a long training run on the server.

In [ ]:
def rollout_frames(friction, mass, n_steps=60, capture_every=15, seed=0):
    """Run a random-action rollout and capture frames at intervals."""
    env = ManipulationEnvV0(headless=True)
    # patch physics
    geom_id = mujoco.mj_name2id(env._model, mujoco.mjtObj.mjOBJ_GEOM, "obj_geom")
    body_id = mujoco.mj_name2id(env._model, mujoco.mjtObj.mjOBJ_BODY, "obj")
    env._model.geom_friction[geom_id, 0] = float(friction)
    env._model.body_mass[body_id] = float(mass)
    s = 0.03
    inertia = mass / 3.0 * (s**2 + s**2)
    env._model.body_inertia[body_id] = np.array([inertia, inertia, inertia])
    mujoco.mj_forward(env._model, env._data)

    # offscreen renderer for frame capture
    renderer = mujoco.Renderer(env._model, height=240, width=320)
    cam = mujoco.MjvCamera()
    cam.azimuth, cam.elevation, cam.distance = 150, -25, 1.2
    cam.lookat[:] = [0.4, 0.0, 0.2]
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE

    obs, _ = env.reset()
    rng = np.random.default_rng(seed)
    frames = []

    for t in range(n_steps):
        action = rng.uniform(-1, 1, size=3).astype(np.float32)
        obs, reward, terminated, truncated, _ = env.step(action)
        if t % capture_every == 0 or terminated or truncated:
            renderer.update_scene(env._data, camera=cam)
            frames.append(renderer.render().copy())
        if terminated or truncated:
            break

    renderer.close()
    return frames


# ── Compare default, target, and extreme settings ──
configs = {
    "default\n(fric=1.0, m=0.1)":  (1.0, 0.10),
    "target\n(fric=0.4, m=0.3)":   (0.4, 0.30),
    "low fric\n(fric=0.15, m=0.1)": (0.15, 0.10),
    "heavy\n(fric=1.0, m=0.45)":   (1.0, 0.45),
}

n_cols = 4  # frames per row
fig, axes = plt.subplots(len(configs), n_cols, figsize=(3.5 * n_cols, 3 * len(configs)))

for row, (label, (fric, mass)) in enumerate(configs.items()):
    frames = rollout_frames(fric, mass, n_steps=60, capture_every=15)
    for col in range(n_cols):
        ax = axes[row, col] if len(configs) > 1 else axes[col]
        if col < len(frames):
            ax.imshow(frames[col])
            ax.set_title(f"t={col * 15}", fontsize=9)
        else:
            ax.axis("off")
        ax.set_xticks([]); ax.set_yticks([])
    axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, labelpad=80, va="center")

fig.suptitle("Random-action rollouts at different physics settings", fontsize=13)
plt.tight_layout()
plt.savefig("manipulation_physics_sanity_check.png", dpi=150)
plt.show()
print("Saved to manipulation_physics_sanity_check.png")

## 9) Smoke Test (Fast)

`T=3`, `blocks=5`, `episodes_per_block=10`, `B_r=5`

In [ ]:
smoke_result = run_bo_doraemon_paperlike(
    checkpoint_path=WARMSTART_CHECKPOINT,
    mu0=MU0_EDGE,
    conc0=C_INIT,
    T=SMOKE_CFG["T"],
    blocks=SMOKE_CFG["blocks"],
    episodes_per_block=SMOKE_CFG["episodes_per_block"],
    B_r=SMOKE_CFG["B_r"],
    bounds=EDGE_DR_BOUNDS,
    max_steps=MAX_EPISODE_STEPS,
    alpha_target=ALPHA_TARGET,
    kl_max=KL_MAX,
    n_candidates=CANDIDATES_PER_UPDATE,
    mean_std=MEAN_PERTURB_STD,
    logc_std=LOGC_PERTURB_STD,
    c_min=C_MIN,
    c_max=C_MAX,
    min_log_w=MIN_WEIGHT_LOG,
    max_log_w=MAX_WEIGHT_LOG,
    target_params=TARGET_PARAMS,
    seed=SEED,
    freeze_doraemon_mean=FREEZE_DORAEMON_MEAN,
    conc_init_mode=CONC_INIT_MODE,
    conc_blend=CONC_BLEND,
)
print()
print(f"Smoke best score:  {smoke_result['best_score']:.3f}")
print(f"Smoke best center: {smoke_result['best_center']}")
print(f"Run dir: {smoke_result['run_dir']}")

## 10) Visualisation: BO Progress and Centre Trajectory

In [ ]:
result = smoke_result
hist = result["history"]

iters = [r["iter"] for r in hist]
fvals = [r["target_solve_rate"] for r in hist]
glast = [r.get("g_last", float("nan")) for r in hist]
mu_f  = [r["mu_friction"] for r in hist]
mu_m  = [r["mu_mass"] for r in hist]
cf    = [r["conc_friction_end"] for r in hist]
cm    = [r["conc_mass_end"] for r in hist]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(iters, fvals, marker="o")
axes[0].set_title("Target solve-rate vs BO iteration")
axes[0].set_xlabel("BO iteration")
axes[0].set_ylabel("Target solve-rate")
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(True, alpha=0.3)

axes[1].plot(iters, glast, marker="o", label="g_last (IS estimate)")
axes[1].axhline(ALPHA_TARGET, color="r", linestyle="--", label="alpha_target")
axes[1].set_title("DORAEMON Success Constraint")
axes[1].set_xlabel("BO iteration")
axes[1].set_ylabel("Success estimate")
axes[1].set_ylim(-0.02, 1.02)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(iters, cf, marker="o", label="conc_friction")
axes[2].plot(iters, cm, marker="o", label="conc_mass")
axes[2].set_title("Beta concentration")
axes[2].set_xlabel("BO iteration")
axes[2].set_ylabel("Concentration")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

sc = axes[3].scatter(mu_f, mu_m, c=iters, cmap="viridis", s=60)
axes[3].plot(mu_f, mu_m, alpha=0.5)
axes[3].set_title("Centre trajectory in (friction, mass)")
axes[3].set_xlabel("friction centre")
axes[3].set_ylabel("mass centre")
axes[3].grid(True, alpha=0.3)
plt.colorbar(sc, ax=axes[3], label="BO iter")

plt.tight_layout()
plt.savefig(Path(result["run_dir"]) / "bo_progress.png", dpi=150)
plt.show()
print(f"Plot saved to {result['run_dir']}/bo_progress.png")